# Session 9 — Your turn: Inference and serving

Two back-of-the-envelope formulas that predict, before measuring anything, how much memory generation needs and how fast it can go.

This is the **solutions** notebook: every task is filled in and every check passes. Try the tasks yourself first.

In [1]:
import sys; sys.path[:0] = ["..", "."]   # aij_check.py: in the folder above, or beside this notebook
import numpy as np
from aij_check import check, close, summary

## Task 1: The size of the KV cache

Write `kv_bytes_per_token(layers, kv_heads, head_dim, bytes_per_number=2)`: for each token every layer stores one key and one value vector per key/value head.
Use it for GPT-2 small (12 layers, 12 heads of 64) and Qwen2.5-0.5B (24 layers, 2 key/value heads of 64), both in 16-bit numbers, and store `gpt2_per_token`, `qwen_per_token`, and `qwen_cache_mb_at_4k`, the Qwen cache in megabytes (1 MB = 10**6 bytes) for a context of 4,096 tokens.

In [2]:
def kv_bytes_per_token(layers, kv_heads, head_dim, bytes_per_number=2):
    return 2 * layers * kv_heads * head_dim * bytes_per_number


gpt2_per_token = kv_bytes_per_token(12, 12, 64)
qwen_per_token = kv_bytes_per_token(24, 2, 64)
qwen_cache_mb_at_4k = qwen_per_token * 4096 / 1e6
print(gpt2_per_token, qwen_per_token, qwen_cache_mb_at_4k)

36864 12288 50.331648


In [3]:
check("GPT-2 stores 36,864 bytes per token", lambda: gpt2_per_token == 36864)
check("Qwen2.5-0.5B stores 12,288 bytes per token, a third as much", lambda: qwen_per_token == 12288)
check("the Qwen cache for 4,096 tokens is about 50 MB", lambda: close(qwen_cache_mb_at_4k, 50.33, 0.01))
check("8-bit numbers halve it", lambda: kv_bytes_per_token(24, 2, 64, 1) == 6144)
summary()

PASS  GPT-2 stores 36,864 bytes per token
PASS  Qwen2.5-0.5B stores 12,288 bytes per token, a third as much
PASS  the Qwen cache for 4,096 tokens is about 50 MB
PASS  8-bit numbers halve it

4 of 4 checks passed.  Well done.


**Why it works.** Grouped-query attention gives Qwen only 2 key/value heads for its 14 query heads, so its cache per token is a third of GPT-2's although the model is bigger. The cache grows linearly with the context and the number of simultaneous requests, and it competes with the weights for memory.

## Task 2: How fast can decoding go?

Decoding one token reads every weight once, so it cannot be faster than the memory bandwidth allows. Write `max_tokens_per_second(weight_bytes, bandwidth_gb_s)` and use it for a 0.99 GB model on a 209 GB/s memory (`bf16_limit`),
the same model with its block weights stored in 4 bits (assume 358 of the 494 million parameters are quantised at 4.125 bits including scales and the other 136 million stay in 16 bits: compute `int4_bytes` and `int4_limit`), and `speedup`.

In [4]:
def max_tokens_per_second(weight_bytes, bandwidth_gb_s):
    return bandwidth_gb_s * 1e9 / weight_bytes


bf16_limit = max_tokens_per_second(0.99e9, 209)
int4_bytes = 358e6 * 4.125 / 8 + 136e6 * 2
int4_limit = max_tokens_per_second(int4_bytes, 209)
speedup = int4_limit / bf16_limit
print(bf16_limit, int4_bytes / 1e9, int4_limit, speedup)

211.11111111111111 0.45659375 457.7373211963589 2.168229416193279


In [5]:
check("the bf16 limit is about 211 tokens per second", lambda: close(bf16_limit, 211.1, 0.5))
check("the int4 model is about 0.46 GB", lambda: close(int4_bytes / 1e9, 0.4566, 0.001))
check("the int4 limit is about twice the bf16 limit, not four times (the embedding stays in 16 bits)", lambda: 1.9 < speedup < 2.3)
check("a faster memory raises the limit in proportion", lambda: close(max_tokens_per_second(1e9, 400) / max_tokens_per_second(1e9, 200), 2.0, 1e-9))
summary()

PASS  the bf16 limit is about 211 tokens per second
PASS  the int4 model is about 0.46 GB
PASS  the int4 limit is about twice the bf16 limit, not four times (the embedding stays in 16 bits)
PASS  a faster memory raises the limit in proportion

4 of 4 checks passed.  Well done.


**Why it works.** Decoding one token must read every weight, so the memory bandwidth sets a ceiling and shrinking the weights lowers it as much as faster memory would. Quantising only the blocks leaves a large 16-bit embedding that caps the gain, which is why 4-bit weights give a bit over 2 times, not 4 times. These are ceilings, not predictions: eager decoding in the session takes 72 ms per token against the 5 ms ceiling, because per-step launch overhead dominates until the model is compiled.